In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('Customer-Churn-Records-.csv', sep=',')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RowNumber           10000 non-null  int64  
 1   CustomerId          10000 non-null  int64  
 2   Surname             10000 non-null  object 
 3   CreditScore         10000 non-null  int64  
 4   Geography           10000 non-null  object 
 5   Gender              10000 non-null  object 
 6   Age                 10000 non-null  int64  
 7   Tenure              10000 non-null  int64  
 8   Balance             10000 non-null  float64
 9   NumOfProducts       10000 non-null  int64  
 10  HasCrCard           10000 non-null  int64  
 11  IsActiveMember      10000 non-null  int64  
 12  EstimatedSalary     10000 non-null  float64
 13  Exited              10000 non-null  int64  
 14  Complain            10000 non-null  int64  
 15  Satisfaction Score  10000 non-null  int64  
 16  Card 

1 - Qual é a taxa geral de Churn dos Clientes ?

In [5]:
churn_rate = df['Exited'].value_counts(normalize=True) * 100
churn_rate

,proportion
Exited,
0,79.62
1,20.38


In [7]:
import plotly.express as px

churn_data = pd.DataFrame({'Staus': ['Permaneceu', 'Saiu'], 'Porcentagem' : churn_rate.values})

fig = px.bar(churn_data, x='Staus', y='Porcentagem',
             title ='Taxa de Churn dos Clientes',
             labels={'Status': 'Status', 'Porcentagem': 'Porcentagem(%)'},
             text_auto=True,
             color='Staus',
             color_discrete_map={'Permaneceu': 'green', 'Saiu': 'red'})

fig.update_layout(yaxis_range=[0,100])
fig.show()

2 - Clientes com reclamações (Complain = 1) têm maior propensão a churn ?

In [8]:
churn_by_complain = df.groupby('Complain')['Exited'].value_counts(normalize=True).unstack().round(4) * 100
churn_by_complain

Exited,0,1
Complain,,
0,99.95,0.05
1,0.49,99.51


In [12]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(name='Permaneceu', x=churn_by_complain.index, y=churn_by_complain[0], marker_color='green'),
    go.Bar(name='Saiu', x=churn_by_complain.index, y=churn_by_complain[1], marker_color='red')
])

fig.update_layout(barmode='group',
                  title='Taxa Churn por Reclamação',
                  yaxis_title= 'Porcentagem(%)',
                  xaxis_title='Complain (0=Sem Reclamação, 1=com reclamação)',
                  xaxis = dict(tickmode = 'array', tickvals = [0,1], ticktext = ['Sem reclamação', 'Com reclamação']),
                  yaxis_range=[0,100]
)

fig.show()


3 - Qual é o perfil médio dos clientes que saíram em termos de idade ?

In [17]:
fig = px.histogram(df, x ='Age',
                   title ='Distribuição de Idade por Clientes',
                   labels = {'Age': 'Idade'}
                   )
fig.show()

In [19]:
age_bins = [18, 30, 40, 50, 60, 100]
age_labels = ['18-29', '30-39', '40-49', '50-59', '60+']
df['AgeGroup'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels, right=False)

churn_by_age_group = df.groupby('AgeGroup')['Exited'].value_counts(normalize=True).unstack().round(4) * 100

age_group_highest_churn = churn_by_age_group[1].idxmax()
highest_churn_rate = churn_by_age_group[1].max()

print("Taxa de Churn por faixa etária:")
print(churn_by_age_group)
print("\nFaixa etária com maior taxa de Churn:")
print(f"\nA Faixa etária: {age_group_highest_churn}, Taxa de Churn: {highest_churn_rate}%")



Taxa de Churn por faixa etária:
Exited        0      1
AgeGroup              
18-29     92.44   7.56
30-39     89.12  10.88
40-49     69.17  30.83
50-59     43.96  56.04
60+       72.05  27.95

Faixa etária com maior taxa de Churn:

A Faixa etária: 50-59, Taxa de Churn: 56.04%


/tmp/ipykernel_1227/758116487.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [20]:
fig = go.Figure(data=[
    go.Bar(name='Permaneceu', x=churn_by_age_group.index, y=churn_by_age_group[0], marker_color='green'),
    go.Bar(name='Saiu', x=churn_by_age_group.index, y=churn_by_age_group[1], marker_color='red')
])

fig.update_layout(barmode='group',
                  title='Taxa Churn por Faixa Etária',
                  yaxis_title= 'Porcentagem(%)',
                  xaxis_title='Faixa Etária',
                  yaxis_range=[0,100]
)

fig.show()

4 - Clientes com o score credito mais baixo estão mais propensos à sair ?

In [23]:
import plotly.graph_objects as go
import pandas as pd

credit_score_bins = [0, 600, 700, 800, 850]
credit_score_labels = ['0-599', '600-699', '700-799', '800+']
df['CreditScoreGroup'] = pd.cut(df['CreditScore'], bins=credit_score_bins, labels=credit_score_labels, right=False)

churn_by_credit_score_group = df.groupby('CreditScoreGroup')['Exited'].value_counts(normalize=True).unstack().round(4) * 100

fig = go.Figure(data=[
    go.Bar(name= 'Permaneceu', x=churn_by_credit_score_group.index, y=churn_by_credit_score_group[0], marker_color='green'),
    go.Bar(name= 'Saiu', x=churn_by_credit_score_group.index, y=churn_by_credit_score_group[1], marker_color='red')
])

fig.update_layout(barmode='group',
                  title='Taxa Churn por Faixa de Score Crédito',
                  yaxis_title= 'Porcentagem(%)',
                  xaxis_title='Faixa de Score de Crédito',
                  yaxis_range=[0,100]
)

fig.show()

/tmp/ipykernel_1227/1719938928.py:8: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



5 - Qual tipo de Cartão tem maior Taxa de Churn ?

In [25]:
churn_by_card_type = df.groupby('Card Type')['Exited'].value_counts(normalize=True).unstack().round(4) * 100

churn_by_type_highest_churn = churn_by_card_type[1].idxmax()
highest_churn_rate = churn_by_card_type[1].max()

print("Taxa de Churn por tipo de cartão:")
print(churn_by_card_type)
print("\nTipo de cartão com maior taxa de Churn:")
print(f"\nO tipo de cartão: {churn_by_type_highest_churn}, Taxa de Churn: {highest_churn_rate}%")

Taxa de Churn por tipo de cartão:
Exited         0      1
Card Type              
DIAMOND    78.22  21.78
GOLD       80.74  19.26
PLATINUM   79.64  20.36
SILVER     79.89  20.11

Tipo de cartão com maior taxa de Churn:

O tipo de cartão: DIAMOND, Taxa de Churn: 21.78%
